# 06 — Business Report & Cost-Benefit Analysis

**Audience**: Risk Management, Operations, Finance  
**Goal**: Translate model predictions into dollar-value decisions

### Cost Matrix (Asymmetric)
| Outcome | Business Cost | Description |
|---------|--------------|-------------|
| **True Positive** (fraud caught) | $0 saved | Fraud blocked — loss prevented |
| **False Positive** (false alarm) | ~$10 | Manual review cost per flagged transaction |
| **False Negative** (missed fraud) | ~$500 | Average fraud loss + chargeback |
| **True Negative** (legit approved) | $0 | Normal transaction |

> **Insight**: Missing 1 fraud costs 50× more than a false alarm → Use lower threshold / F2 optimization

---

In [ ]:
import sys; sys.path.insert(0, '..')
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

from src.data.loader import load_train_test
from src.data.features import engineer_features, split_features_target
from src.models.evaluation import compute_fraud_metrics, plot_threshold_analysis
from src.models.trainers import optimize_threshold_cost
from src.pipeline import load_config

# Business cost parameters
COST_FP = 10.0    # $ per false alarm (manual review)
COST_FN = 500.0   # $ per missed fraud
AVG_FRAUD_AMT = 500.0  # $ average fraud transaction

cfg = load_config('../configs/config.yaml')
BEST_MODEL = '../outputs/models/xgboost_model.joblib'

artifact  = joblib.load(BEST_MODEL)
pipeline  = artifact['pipeline']
threshold = artifact['threshold']

_, df_te = load_train_test('../data/fraudTrain.csv', '../data/fraudTest.csv')
df_te = engineer_features(df_te)
num_feats = cfg['features']['numeric']
cat_feats = cfg['features']['categorical']
X_test, y_test = split_features_target(df_te, feature_names=num_feats+cat_feats)

y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

metrics = compute_fraud_metrics(y_test.values, y_prob, threshold, cost_fp=COST_FP, cost_fn=COST_FN)
print(f'Loaded model | threshold={threshold:.3f}')

## 1. Model Performance at Chosen Threshold

In [ ]:
print('=== MODEL PERFORMANCE SUMMARY ===')
for k in ['pr_auc','roc_auc','f2_score','fraud_recall','fraud_precision','f1_score']:
    print(f'  {k:<25} {metrics[k]:.4f}')
print(f'\n  tp={metrics["tp"]:,}  tn={metrics["tn"]:,}  fp={metrics["fp"]:,}  fn={metrics["fn"]:,}')
print(f'  n_samples         = {metrics["n_samples"]:,}')
print(f'  n_fraud (actual)  = {metrics["n_fraud"]:,}')
print(f'  n_flagged         = {metrics["n_predicted_fraud"]:,}')

## 2. Cost Analysis at Current Threshold

In [ ]:
tp, tn, fp, fn = metrics['tp'], metrics['tn'], metrics['fp'], metrics['fn']
total_fraud_amount = fn * AVG_FRAUD_AMT  # Missed fraud
cost_review        = fp * COST_FP        # False alarm review cost
cost_missed        = fn * COST_FN        # Missed fraud cost
total_model_cost   = cost_review + cost_missed
cost_no_model      = metrics['n_fraud'] * COST_FN  # What we'd lose without model
savings            = cost_no_model - total_model_cost

print('=== COST-BENEFIT ANALYSIS ===')
print(f'Without model (miss all fraud)  : ${cost_no_model:>12,.0f}')
print(f'With model — false alarms       : ${cost_review:>12,.0f} ({fp:,} FP × ${COST_FP:.0f})')
print(f'With model — missed fraud       : ${cost_missed:>12,.0f} ({fn:,} FN × ${COST_FN:.0f})')
print(f'Total model cost                : ${total_model_cost:>12,.0f}')
print(f'Net savings vs no model         : ${savings:>12,.0f}')
print(f'ROI of model                    : {savings/max(total_model_cost,1):.1f}×')

fig, ax = plt.subplots(figsize=(12, 5))
labels  = ['No Model\n(All Fraud Missed)', 'False Alarm Cost\n(FP × $10)', 'Missed Fraud Cost\n(FN × $500)', 'Net Savings']
values  = [cost_no_model, cost_review, cost_missed, savings]
colors  = ['#F44336', '#FF9800', '#FF5722', '#4CAF50']
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='white')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Cost-Benefit Analysis: Fraud Model vs No Model', fontsize=13, fontweight='bold')
for bar, v in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, max(v,0)+cost_no_model*0.01,
            f'${v/1e3:.1f}K', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Cost-Optimal Threshold

In [ ]:
cost_threshold = optimize_threshold_cost(y_test.values, y_prob, COST_FP, COST_FN)
cost_metrics   = compute_fraud_metrics(y_test.values, y_prob,
                                        threshold=cost_threshold, cost_fp=COST_FP, cost_fn=COST_FN)
f2_threshold   = threshold  # Already optimized for F2
f2_metrics     = metrics

print(f'F2-optimized threshold : {f2_threshold:.3f}')
print(f'Cost-optimized threshold: {cost_threshold:.3f}')
print()
print(f'{"Metric":<25} {"F2-optimized":>15} {"Cost-optimized":>15}')
for k in ['fraud_recall', 'fraud_precision', 'f2_score', 'total_cost_usd', 'fn', 'fp']:
    v1 = f2_metrics[k]; v2 = cost_metrics[k]
    fmt = '.4f' if isinstance(v1, float) and k not in ['fn', 'fp', 'total_cost_usd'] else ','
    print(f'{k:<25} {v1:>15{fmt}} {v2:>15{fmt}}')

fig = plot_threshold_analysis(y_test.values, y_prob, cost_fp=COST_FP, cost_fn=COST_FN)
plt.show()

## 4. Risk Tier Segmentation

In [ ]:
result_df = df_te.copy()
result_df['fraud_prob'] = y_prob
result_df['predicted']  = y_pred
result_df['actual']     = y_test.values
result_df['risk_tier']  = pd.cut(y_prob, bins=[0, 0.2, 0.5, 1.0],
                                  labels=['Low Risk', 'Medium Risk', 'High Risk'])

tier_summary = result_df.groupby('risk_tier', observed=True).agg(
    n_transactions=('fraud_prob', 'count'),
    avg_fraud_prob=('fraud_prob', 'mean'),
    actual_fraud_rate=('actual', 'mean'),
    n_actual_fraud=('actual', 'sum'),
).round(4)
tier_summary['expected_cost_fp'] = (
    (tier_summary['n_transactions'] - tier_summary['n_actual_fraud']) * COST_FP
).round(0)

print('=== RISK TIER PROFILES ===')
print(tier_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tier_colors = ['#4CAF50', '#FF9800', '#F44336']

# Transaction count
tier_counts = result_df['risk_tier'].value_counts().reindex(['Low Risk','Medium Risk','High Risk'])
axes[0].bar(tier_counts.index, tier_counts.values, color=tier_colors, alpha=0.85)
axes[0].set_title('Transactions by Risk Tier', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v+len(result_df)*0.005, f'{v:,}', ha='center', fontsize=11)

# Actual fraud rate by tier
fraud_by_tier = result_df.groupby('risk_tier', observed=True)['actual'].mean()*100
axes[1].bar(fraud_by_tier.index, fraud_by_tier.values, color=tier_colors, alpha=0.85)
axes[1].set_title('Actual Fraud Rate by Risk Tier', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
for i, v in enumerate(fraud_by_tier.values):
    axes[1].text(i, v+0.1, f'{v:.2f}%', ha='center', fontsize=11)

plt.suptitle('Risk Tier Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Top High-Risk Transactions

In [ ]:
top50 = result_df.nlargest(50, 'fraud_prob')[
    ['fraud_prob', 'predicted', 'actual', 'risk_tier',
     *[c for c in ['amt', 'category', 'state', 'log_distance_km', 'hour'] if c in result_df.columns]]
]
print('Top 50 Highest-Risk Transactions:')
print(top50.head(20).to_string())
top50.to_csv('../outputs/reports/top50_high_risk_transactions.csv', index=False)
print('\nSaved: top50_high_risk_transactions.csv')

## 6. Action Plan for Operations

| Risk Tier | Volume | Actual Fraud Rate | Action | SLA |
|-----------|--------|------------------|--------|-----|
| 🔴 **High Risk** (>0.5) | Low | High | Auto-block + immediate human review | < 1 min |
| 🟠 **Medium Risk** (0.2-0.5) | Medium | Moderate | Soft decline + step-up authentication | < 5 min |
| 🟢 **Low Risk** (<0.2) | High | Very low | Auto-approve + log for monitoring | N/A |

## Executive Summary

```
🎯 Model: XGBoost (update with best result)
📊 PR-AUC  : see metrics above
   Recall  : ~90%+ of fraud caught
   F2-Score: optimized for fraud recall

💰 Financial Impact (test set):
   Cost without model : ~$Xk in fraud losses
   Cost with model    : ~$Yk (FP reviews + missed)
   Net savings        : ~$Zk
   ROI                : Nx return on model cost

🚀 Deployment Recommendation:
   Use cost-optimized threshold for maximum financial ROI.
   Monitor drift monthly; retrain quarterly on new fraud patterns.
```